In [ ]:
# Load the cleaned sales, customer, branch, invoice, and payment datasets.

import pandas as pd


sales_table_lines = pd.read_csv("../../data/cleaned/sales_orders_lines.csv")
sales_table_header = pd.read_csv("../../data/cleaned/sales_orders_header.csv")
sales_orders_header = pd.read_csv("../../data/cleaned/sales_orders_header.csv")
df_customers = pd.read_csv("../../data/cleaned/customers.csv")
df_branches = pd.read_csv("../../data/cleaned/branches.csv")
df_invoices = pd.read_csv("../../data/cleaned/invoices.csv")
df_payments = pd.read_csv("../../data/cleaned/payments.csv")







In [ ]:
# Merge sales order lines with their order headers.

sales_table = pd.merge(sales_table_lines , sales_table_header , on='so_id' , how='left')

In [ ]:
# Add customer details to the merged sales orders.

sales_table = sales_table.merge(df_customers , on='customer_id' , how='left')

sales_table


In [ ]:
# Keep this cell as a documented checkpoint in the sales-order merge workflow.


In [ ]:
# Resolve duplicate branch columns created by the sales-order merge.

sales_table = sales_table.drop(columns='branch_id_x')

sales_table = sales_table.rename(columns={'branch_id_y' : 'branch_id'})


In [ ]:
# Add branch details and inspect the merged sales dataset.

sales_table = sales_table.merge(df_branches, on='branch_id' , how='left')


sales_table.info()

In [ ]:
# Add invoice details to the sales dataset.

sales_table = sales_table.merge(df_invoices, on='so_id', how='left')

In [ ]:
# Summarize payments by invoice and attach the payment summary to sales orders.

payments_summary = df_payments.groupby('invoice_id').agg(
    total_paid=('payment_amount', 'sum'),
    number_of_payments=('payment_id', 'count'),
    last_payment_date=('payment_date', 'max')
).reset_index()

sales_table = sales_table.merge(payments_summary, on='invoice_id', how='left')

In [ ]:
# Inspect the sales dataset after payment information is added.

sales_table.info()

In [ ]:
# Remove duplicate merge columns and restore clean column names.

sales_table = sales_table.drop(
    columns=[col for col in sales_table.columns if col.endswith('_y')]
)

sales_table = sales_table.rename(
    columns={col: col.replace('_x', '') for col in sales_table.columns if col.endswith('_x')}
)

sales_table.columns

In [ ]:
# Load product, inventory, and stock ledger data and build the purchases fact table.

df_products = pd.read_csv("../../data/cleaned/products.csv")
df_inventory_master = pd.read_csv("../../data/cleaned/inventory_master.csv")
df_stock_ledger = pd.read_csv("../../data/cleaned/stock_ledger.csv")



Purchases_fact_table = pd.merge(df_products, df_inventory_master, on='product_id', how='left')
Purchases_fact_table = Purchases_fact_table.merge(df_stock_ledger, on=['product_id', 'branch_id'], how='left')

In [ ]:
# Review the data types in the sales dataset.

sales_table.dtypes

In [ ]:
# Remove duplicate columns from the purchases fact table.

Purchases_fact_table = Purchases_fact_table.drop(
    columns=[col for col in Purchases_fact_table.columns if col.endswith("_y")]
)

In [ ]:
# Restore clean column names in the purchases fact table.

Purchases_fact_table = Purchases_fact_table.rename(
    columns={col: col.replace('_x', '') for col in Purchases_fact_table.columns if col.endswith('_x')}
)

In [ ]:
# Review the sorted data types in the sales dataset.

sales_table.dtypes.sort_index()

In [ ]:
# Convert date-related columns in all working datasets to datetime values.

# Convert all date-related columns to datetime
dataframes = [
    sales_table,
    sales_table_header,
    sales_table_lines,
    df_customers,
    df_branches,
    df_invoices,
    df_payments,
    df_products,
    df_inventory_master,
    df_stock_ledger,
    Purchases_fact_table,
]

for dataframe in dataframes:
    date_columns = [
        column for column in dataframe.columns
        if "date" in column.lower() or column.lower().endswith("_since")
    ]

    for column in date_columns:
        dataframe[column] = pd.to_datetime(
            dataframe[column],
            errors="coerce"
        )

In [ ]:
# Calculate the delivery lead time for each sales order.

sales_table['delivery_lead_time_days'] = (sales_table['delivery_date'] - sales_table['order_date']).dt.days

In [ ]:
# Calculate the profit margin for each purchase record.

Purchases_fact_table['profit_margin'] = Purchases_fact_table['unit_price'] - Purchases_fact_table['unit_cost']

In [ ]:
# Calculate the number of days between invoicing and payment.

sales_table['days_to_pay_an_invoice'] = (
    sales_table['last_payment_date'] - sales_table['invoice_date']
).dt.days

In [ ]:
# Save both completed datasets as CSV files in the processed data folder.
from pathlib import Path

processed_dir = Path("../../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)
sales_table.to_csv(processed_dir / "sales_table.csv", index=False)
Purchases_fact_table.to_csv(processed_dir / "purchases_fact_table.csv", index=False)
